# 12 — Continuous latent actions from video

This lesson compares a bounded continuous action inferred from two frames with the discrete codes in Notebook 07.

## Learning goals

- Infer a compact action-like value from a pair of frames.
- Compare continuous values with discrete codes on the same clips.
- Test transfer to a new shape and a changed camera.
- Separate this small local measurement from the paper's findings.

## Paper context

The paper reports that continuous sparse or noisy latents can represent a wider range of prediction capacity than vector quantization. It also reports action transfer between different objects, with transformations that remain local to the camera view. The paper uses a frozen V-JEPA 2-L encoder, a 128-dimensional latent, and YoutubeTemporal-1B. Its default setup uses 16-frame clips at 4 fps, 30,000 training steps, and batch size 1,024. Appendix A reports about 12 hours on 64 H100 GPUs. See the [version 2 paper](https://arxiv.org/html/2601.05230v2).

This notebook is a synthetic teaching analogue. It renders simple shapes, measures fixed image moments, and trains small models on a CPU. It does not use real video, a learned visual encoder, the paper's data or code, or robot hardware. It does not reproduce the paper or validate real-world transfer. The 2026-10-14 paper source review did not confirm first-party code, a released checkpoint, or usable training-data access.

In [ ]:
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)
device = torch.device("cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Platform:", platform.platform())
print("Device:", device)

## 1. Render short clips and extract fixed frame features

Each clip has one white shape on a black background. The frame feature uses the foreground centroid and second moments. This fixed image summary makes the example small and easy to inspect. It is not a learned visual feature.

In [ ]:
FRAME_SIZE = 32
DATA_SEED = 20261014
TRAIN_SEEDS = [7, 17, 27, 37, 47]
SHAPES = ("circle", "square", "triangle")
CAMERA_A = {"name": "identity", "angle": 0.0, "scale": 1.0, "offset": np.array([0.0, 0.0])}
CAMERA_B = {
    "name": "rotated_zoomed",
    "angle": np.deg2rad(25.0),
    "scale": 1.15,
    "offset": np.array([0.04, -0.03]),
}
FEATURE_NAMES = ("centroid_x", "centroid_y", "variance_x", "variance_y", "covariance_xy")
GRID = np.linspace(-1.0, 1.0, FRAME_SIZE, dtype=np.float32)
GRID_X, GRID_Y = np.meshgrid(GRID, GRID)


def camera_point(world_point, camera):
    cosine = np.cos(camera["angle"])
    sine = np.sin(camera["angle"])
    rotation = np.array([[cosine, -sine], [sine, cosine]], dtype=np.float32)
    return camera["scale"] * (rotation @ np.asarray(world_point, dtype=np.float32)) + camera["offset"]


def render_frame(world_point, shape, camera):
    center_x, center_y = camera_point(world_point, camera)
    cosine = np.cos(camera["angle"])
    sine = np.sin(camera["angle"])
    dx_screen = GRID_X - center_x
    dy_screen = GRID_Y - center_y
    dx = (cosine * dx_screen + sine * dy_screen) / camera["scale"]
    dy = (-sine * dx_screen + cosine * dy_screen) / camera["scale"]
    radius = 0.22
    margin = radius * camera["scale"] * np.sqrt(2.0)
    if abs(center_x) + margin > 1.0 or abs(center_y) + margin > 1.0:
        raise ValueError("Camera transform crops the object outside the frame")
    if shape == "circle":
        mask = dx**2 + dy**2 <= radius**2
    elif shape == "square":
        mask = np.maximum(np.abs(dx), np.abs(dy)) <= radius
    elif shape == "triangle":
        half_width = np.maximum((radius - dy) / 2.0, 0.0)
        mask = (dy >= -radius) & (dy <= radius) & (np.abs(dx) <= half_width)
    else:
        raise ValueError(f"Unknown shape: {shape}")
    return mask.astype(np.float32)


def frame_moments(frame):
    y_index, x_index = np.nonzero(frame > 0.5)
    if len(x_index) == 0:
        raise ValueError("Frame has no visible foreground pixels")
    x = (x_index.astype(np.float32) - (FRAME_SIZE - 1) / 2) / ((FRAME_SIZE - 1) / 2)
    y = (y_index.astype(np.float32) - (FRAME_SIZE - 1) / 2) / ((FRAME_SIZE - 1) / 2)
    centered_x = x - x.mean()
    centered_y = y - y.mean()
    return np.array(
        [x.mean(), y.mean(), np.mean(centered_x**2), np.mean(centered_y**2),
         np.mean(centered_x * centered_y)],
        dtype=np.float32,
    )


def render_clip(start, displacement, shape, camera):
    positions = [np.asarray(start) + step * np.asarray(displacement) for step in range(3)]
    frames = np.stack([render_frame(position, shape, camera) for position in positions])
    features = np.stack([frame_moments(frame) for frame in frames])
    return frames, features


def generate_dataset(count, seed, shapes, camera, id_start):
    rng = np.random.default_rng(seed)
    starts = rng.uniform(-0.25, 0.25, size=(count, 2)).astype(np.float32)
    displacements = rng.uniform(-0.07, 0.07, size=(count, 2)).astype(np.float32)
    shape_choices = rng.choice(shapes, size=count)
    frames = np.empty((count, 3, FRAME_SIZE, FRAME_SIZE), dtype=np.float32)
    features = np.empty((count, 3, len(FEATURE_NAMES)), dtype=np.float32)
    for index, (start, displacement, shape) in enumerate(zip(starts, displacements, shape_choices)):
        frames[index], features[index] = render_clip(start, displacement, str(shape), camera)
    ids = np.arange(id_start, id_start + count)
    return {"frames": frames, "features": features, "ids": ids, "shapes": shape_choices}


def generate_matched_transfer(count, seed):
    rng = np.random.default_rng(seed)
    starts = rng.uniform(-0.25, 0.25, size=(count, 2)).astype(np.float32)
    displacements = rng.uniform(-0.07, 0.07, size=(count, 2)).astype(np.float32)
    source_frames = np.empty((count, 3, FRAME_SIZE, FRAME_SIZE), dtype=np.float32)
    source_features = np.empty((count, 3, len(FEATURE_NAMES)), dtype=np.float32)
    shape_frames = np.empty_like(source_frames)
    shape_features = np.empty_like(source_features)
    camera_frames = np.empty_like(source_frames)
    camera_features = np.empty_like(source_features)
    for index, (start, displacement) in enumerate(zip(starts, displacements)):
        source_frames[index], source_features[index] = render_clip(start, displacement, "circle", CAMERA_A)
        shape_frames[index], shape_features[index] = render_clip(start, displacement, "triangle", CAMERA_A)
        camera_frames[index], camera_features[index] = render_clip(start, displacement, "circle", CAMERA_B)
    return {
        "source_frames": source_frames,
        "source_features": source_features,
        "shape_frames": shape_frames,
        "shape_features": shape_features,
        "camera_frames": camera_frames,
        "camera_features": camera_features,
    }


train_data = generate_dataset(1024, DATA_SEED, ("circle", "square"), CAMERA_A, 0)
test_data = generate_dataset(256, DATA_SEED + 1, ("circle", "square"), CAMERA_A, 10000)
transfer_data = generate_matched_transfer(128, DATA_SEED + 2)

assert not set(train_data["ids"]) & set(test_data["ids"])
assert np.isfinite(train_data["features"]).all()
assert np.isfinite(test_data["features"]).all()
assert np.isfinite(transfer_data["source_features"]).all()
assert (train_data["frames"].sum(axis=(2, 3)) > 0).all()
assert (test_data["frames"].sum(axis=(2, 3)) > 0).all()

print("Data seed:", DATA_SEED)
print("Training clips:", len(train_data["ids"]), "held-out clips:", len(test_data["ids"]))
print("Transfer cases:", len(transfer_data["source_features"]))
print("Training shapes:", sorted(str(value) for value in set(train_data["shapes"])))
print("Camera views: identity for training; identity, new shape, and rotated/zoomed view for evaluation")
print("Frame feature names:", FEATURE_NAMES)

## 2. Inspect the generated clips

The transfer clips use the same generated starting positions and motions. Only the object shape or camera changes. Motion values are used to match clips. The models do not receive those values.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(8, 12))
examples = [
    (train_data["frames"][0], "Training: circle"),
    (train_data["frames"][1], "Training: square"),
    (test_data["frames"][0], "Held out: identity view"),
    (transfer_data["source_frames"][0], "Source: circle"),
    (transfer_data["shape_frames"][0], "Transfer: triangle"),
    (transfer_data["camera_frames"][0], "Transfer: new camera"),
]
for row, (clip, label) in enumerate(examples):
    for step in range(3):
        axes[row, step].imshow(clip[step], cmap="gray", vmin=0, vmax=1)
        axes[row, step].axis("off")
        if step == 0:
            axes[row, step].set_ylabel(label, fontsize=8)
plt.suptitle("Three frames from each generated clip")
plt.tight_layout()
plt.show()

print("Example frame moments:")
for name, features in [
    ("circle", train_data["features"][0]),
    ("square", train_data["features"][1]),
    ("triangle", transfer_data["shape_features"][0]),
]:
    print(name, np.round(features[0], 3))

## 3. Define the latent-action models

The inverse model sees current and next frame features and infers an action. The forward model uses the current feature and action to predict the next feature. Seeing the future is a training-time inference step; it is not a deployable action policy.

In [ ]:
class ContinuousInverse(nn.Module):
    def __init__(self, feature_dim, latent_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * feature_dim, 64),
            nn.Tanh(),
            nn.Linear(64, latent_dim),
            nn.Tanh(),
        )

    def forward(self, current, next_frame):
        return self.net(torch.cat([current, next_frame], dim=-1))


class DiscreteInverse(nn.Module):
    def __init__(self, feature_dim, code_count=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * feature_dim, 64),
            nn.Tanh(),
            nn.Linear(64, code_count),
        )

    def forward(self, current, next_frame):
        return self.net(torch.cat([current, next_frame], dim=-1))


class ForwardModel(nn.Module):
    def __init__(self, feature_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim + action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, feature_dim),
        )

    def forward(self, current, action):
        return self.net(torch.cat([current, action], dim=-1))


BATCH_SIZE = 128
EPOCHS = 20
LEARNING_RATE = 0.001
CODE_COUNT = 8


def standardize_features(train_features):
    mean = train_features.mean(dim=0, keepdim=True)
    scale = train_features.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-3)
    return mean, scale


def train_method(method, seed):
    torch.manual_seed(seed)
    train_features = torch.tensor(train_data["features"], dtype=torch.float32)
    current_raw = train_features[:, 0]
    next_raw = train_features[:, 1]
    mean, scale = standardize_features(torch.cat([current_raw, next_raw], dim=0))
    current = (current_raw - mean) / scale
    next_frame = (next_raw - mean) / scale
    feature_dim = current.shape[-1]

    if method == "continuous":
        inverse = ContinuousInverse(feature_dim).to(device)
        action_dim = 2
    elif method == "discrete":
        inverse = DiscreteInverse(feature_dim, CODE_COUNT).to(device)
        action_dim = CODE_COUNT
    else:
        raise ValueError(f"Unknown method: {method}")
    forward = ForwardModel(feature_dim, action_dim).to(device)
    optimizer = torch.optim.Adam(
        list(inverse.parameters()) + list(forward.parameters()),
        lr=LEARNING_RATE,
    )
    order_generator = torch.Generator().manual_seed(seed + 10000)
    last_loss = float("nan")

    for _ in range(EPOCHS):
        order = torch.randperm(len(current), generator=order_generator)
        for batch_ids in order.split(BATCH_SIZE):
            current_batch = current[batch_ids]
            next_batch = next_frame[batch_ids]
            if method == "continuous":
                action = inverse(current_batch, next_batch)
            else:
                logits = inverse(current_batch, next_batch)
                action = F.gumbel_softmax(logits, tau=0.8, hard=True)
            prediction = forward(current_batch, action)
            prediction_loss = F.mse_loss(prediction, next_batch)

            if method == "continuous":
                action_std = action.std(dim=0, unbiased=False)
                variance_floor = F.relu(0.05 - action_std).mean()
                regularization = 0.002 * action.abs().mean() + 0.02 * variance_floor
                loss = prediction_loss + regularization
            else:
                probabilities = F.softmax(logits, dim=-1).mean(dim=0)
                entropy = -(probabilities * (probabilities + 1e-8).log()).sum()
                loss = prediction_loss - 0.002 * entropy

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            last_loss = float(loss.detach())

    return {
        "method": method,
        "seed": seed,
        "inverse": inverse.eval(),
        "forward": forward.eval(),
        "mean": mean,
        "scale": scale,
        "training_loss": last_loss,
    }


def infer_action(run, current_raw, next_raw):
    mean, scale = run["mean"], run["scale"]
    current = (current_raw - mean) / scale
    next_frame = (next_raw - mean) / scale
    with torch.no_grad():
        if run["method"] == "continuous":
            return run["inverse"](current, next_frame)
        logits = run["inverse"](current, next_frame)
        return F.one_hot(logits.argmax(dim=-1), num_classes=CODE_COUNT).float()


def predict_next(run, current_raw, action):
    current = (current_raw - run["mean"]) / run["scale"]
    with torch.no_grad():
        prediction = run["forward"](current, action)
    return prediction


def feature_mse(run, current_raw, next_raw):
    action = infer_action(run, current_raw, next_raw)
    prediction = predict_next(run, current_raw, action)
    target = (next_raw - run["mean"]) / run["scale"]
    return float((prediction - target).square().mean().item())


def transfer_mse(run, source_features, target_features):
    source_current = torch.tensor(source_features[:, 0], dtype=torch.float32)
    source_next = torch.tensor(source_features[:, 1], dtype=torch.float32)
    target_current = torch.tensor(target_features[:, 0], dtype=torch.float32)
    target_next = torch.tensor(target_features[:, 1], dtype=torch.float32)
    action = infer_action(run, source_current, source_next)
    prediction = predict_next(run, target_current, action)
    target = (target_next - run["mean"]) / run["scale"]
    return float((prediction - target).square().mean().item())

## 4. Train both methods on the same clips

The continuous latent is bounded by tanh and has a small sparsity and variance-floor penalty. The discrete baseline uses eight hard Gumbel-Softmax codes. Both use the same split, training settings, batch order per seed, and five seeds. Their action dimensions and objectives differ, so this is not a capacity-matched study.

In [ ]:
runs = []
run_errors = []
started = time.time()
for seed in TRAIN_SEEDS:
    for method in ("continuous", "discrete"):
        try:
            run = train_method(method, seed)
            test_features = torch.tensor(test_data["features"], dtype=torch.float32)
            run["same_view_mse"] = feature_mse(
                run, test_features[:, 0], test_features[:, 1]
            )
            source = transfer_data["source_features"]
            run["new_shape_mse"] = transfer_mse(
                run, source, transfer_data["shape_features"]
            )
            run["new_camera_mse"] = transfer_mse(
                run, source, transfer_data["camera_features"]
            )
            observed = [run["training_loss"], run["same_view_mse"], run["new_shape_mse"], run["new_camera_mse"]]
            if not np.isfinite(observed).all():
                raise FloatingPointError("Run produced a non-finite loss or metric")
            run["status"] = "complete"
            runs.append(run)
        except Exception as exc:
            run_errors.append({"method": method, "seed": seed, "error": repr(exc)})
            print("FAILED", method, "seed", seed, repr(exc))
elapsed_seconds = time.time() - started

assert len(runs) + len(run_errors) == len(TRAIN_SEEDS) * 2
assert all(np.isfinite([
    run["same_view_mse"], run["new_shape_mse"], run["new_camera_mse"]
]).all() for run in runs)

print(f"Attempted runs: {len(TRAIN_SEEDS)} seeds x 2 methods = {len(TRAIN_SEEDS) * 2}")
print("Completed:", len(runs), "Failed:", len(run_errors))
print("Elapsed seconds:", round(elapsed_seconds, 2))
for run in runs:
    print(
        run["method"], "seed", run["seed"],
        "train loss", round(run["training_loss"], 5),
        "same-view MSE", round(run["same_view_mse"], 5),
        "new-shape MSE", round(run["new_shape_mse"], 5),
        "new-camera MSE", round(run["new_camera_mse"], 5),
    )

## 5. Summarize the measurements

Each score is the mean squared error over normalized frame-feature dimensions and held-out cases. The 95% interval is across five training seeds. It is uncertain because five runs are few.

In [ ]:
T_CRITICAL_DF4 = 2.776
METRICS = ("same_view_mse", "new_shape_mse", "new_camera_mse")


def summarize(values):
    values = np.asarray(values, dtype=np.float64)
    mean = float(values.mean())
    median = float(np.median(values))
    sd = float(values.std(ddof=1)) if len(values) > 1 else float("nan")
    half_width = T_CRITICAL_DF4 * sd / np.sqrt(len(values)) if len(values) == 5 else float("nan")
    return {
        "n": len(values),
        "mean": mean,
        "median": median,
        "sd": sd,
        "ci_low": float(mean - half_width),
        "ci_high": float(mean + half_width),
    }


for method in ("continuous", "discrete"):
    method_runs = [run for run in runs if run["method"] == method]
    print("\n", method, "complete runs:", len(method_runs))
    for metric in METRICS:
        summary = summarize([run[metric] for run in method_runs])
        print(
            metric,
            "per seed =", np.round([run[metric] for run in method_runs], 5),
            "mean =", round(summary["mean"], 5),
            "median =", round(summary["median"], 5),
            "SD =", round(summary["sd"], 5),
            "uncertain 95% t interval =",
            (round(summary["ci_low"], 5), round(summary["ci_high"], 5)),
            "n =", summary["n"],
        )

## 6. Inspect latent use and transfer

The new-camera score applies an action inferred in the identity view to a matched clip from the rotated and zoomed view. A difference can show that the representation depends on camera coordinates in this generated setup. It does not establish what a model will do with natural video.

In [ ]:
for method in ("continuous", "discrete"):
    print("\n", method, "latent-use summaries")
    for run in [item for item in runs if item["method"] == method]:
        source = torch.tensor(transfer_data["source_features"][:, :2], dtype=torch.float32)
        action = infer_action(run, source[:, 0], source[:, 1]).cpu().numpy()
        if method == "continuous":
            print(
                "seed", run["seed"],
                "min", np.round(action.min(axis=0), 3),
                "max", np.round(action.max(axis=0), 3),
                "std", np.round(action.std(axis=0), 3),
            )
            assert np.abs(action).max() <= 1.0 + 1e-6
        else:
            counts = np.bincount(action.argmax(axis=1), minlength=CODE_COUNT)
            print("seed", run["seed"], "code counts", counts.tolist())

plot_metrics = [("same_view_mse", "Same view"), ("new_shape_mse", "New shape"), ("new_camera_mse", "New camera")]
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(plot_metrics))
width = 0.34
for offset, method in enumerate(("continuous", "discrete")):
    means = [np.mean([run[key] for run in runs if run["method"] == method]) for key, _ in plot_metrics]
    sds = [np.std([run[key] for run in runs if run["method"] == method], ddof=1) for key, _ in plot_metrics]
    ax.bar(x + (offset - 0.5) * width, means, width, yerr=sds, capsize=4, label=method)
ax.set_xticks(x, [label for _, label in plot_metrics])
ax.set_ylabel("Normalized feature MSE")
ax.set_title("Mean across five training seeds; bars show SD")
ax.legend()
plt.tight_layout()
plt.show()

## What the evidence supports

This run measures a small synthetic task with one foreground object, a fixed image-moment feature, two camera transforms, and two simple model types. It does not use the paper's data, visual encoder, training scale, evaluation metrics, or planning tests. The model sees future frames when it infers actions, which limits how the action can be used at test time.

The paper's camera-relative result motivates the transfer test. A new shape and a changed camera are useful controlled checks, but they do not represent the range of objects, motion, background changes, or camera motion in real video. This notebook cannot validate the paper's findings.

## Exercises

1. Change the camera rotation or zoom and rerun the fixed seeds.
2. Add a background distractor and measure whether frame moments still work.
3. Replace the fixed moment extractor with a learned frame encoder, then keep the same transfer split.
4. Review the [paper-reproduction matrix](../docs/REPRODUCTION_MATRIX.md) before attempting an exact paper result.